# ML-03 — Frame the Freestyle Lane as an ML Task

## 1. Task type

This is unsupervised page-pair clustering plus within-cluster ranking. Clustering discovers
interaction patterns; it does not learn an is-cannibalizing target. Action names are assigned
only after cluster profiles are inspected.

## 2. Target and output

There is no supervised label. The learned output is cluster_id. Names such as possible
substitution, fragmented shared demand, and complementary overlap are cautious post-hoc
interpretations. A later review score ranks evidence-rich pairs; it is not a probability.

## 3. Success criteria

Use stability across seeds and bootstrap samples, silhouette score, materially different
profiles, sensitivity checks, and structured Top-20 review. Later compare ranking with a
transparent overlap-only baseline. A mathematically neat but non-actionable solution fails.

In [1]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
pd.DataFrame({"stage":["generation","features","learning","interpretation","ranking"],
"grain":["page-query","page pair","page pair","cluster","page pair"],
"output":["shared pairs","evidence vector","cluster_id","review action","priority/reasons"]})

,stage,grain,output
0,generation,page-query,shared pairs
1,features,page pair,evidence vector
2,learning,page pair,cluster_id
3,interpretation,cluster,review action
4,ranking,page pair,priority/reasons


## 4. Candidate population

Development uses published non-deleted pages with at least 500 impressions, their top 50 visible
queries, queries appearing on 2–10 pages per client, and at least two shared hashes. These are
tractability and evidence-floor policies, not universal SEO truths.

In [2]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
con.sql(f"""SELECT COUNT(*) pairs,COUNT(DISTINCT client_hash_id) clients,
MIN(shared_query_count) min_shared_queries,
MEDIAN(total_impressions_a+total_impressions_b) median_pair_impressions FROM {R}""").df()

,pairs,clients,min_shared_queries,median_pair_impressions
0,362562,43,2,12605.0


## 5. Why ML may help

One overlap rule cannot distinguish growth, substitution, joint decline, dominance, balanced
fragmentation, and unique demand without brittle thresholds. Clustering earns its place only
when these combinations form stable, interpretable groups.